# Recommending the next trip
**Part II of II.** [Part I](01_travel_styles_clustering.ipynb) groups European cities into travel styles.

**Audience:** a travel agency that wants to send past customers a personalised offer for their next destination.
**Approach:** item-based collaborative filtering, based on which places travellers visit together.

# Step 0 : SETUP

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
trips = pd.read_csv("https://sds-aau.github.io/SDS-master/M1/data/trips.csv")
print(trips.shape)
print("Travellers:", trips["username"].nunique())
print("Places:", trips["place_slug"].nunique())
trips[["username", "place", "place_slug", "country", "date_start", "date_end"]].head()

(46510, 11)
Travellers: 2871
Places: 960


,username,place,place_slug,country,date_start,date_end
0,@lewellenmichael,Guanajuato,mexico,Mexico,2018-06-04,2018-06-15
1,@lewellenmichael,Mexico City,mexico-city-mexico,Mexico,2018-05-31,2018-06-03
2,@lewellenmichael,Cancun,cancun-mexico,Mexico,2017-11-01,2017-11-05
3,@lewellenmichael,Amman,amman-jordan,Jordan,2017-07-24,2017-08-07
4,@waylandchin,Yingkou,china,China,2017-02-17,2017-03-18


## 1. Cleaning the data

Before building the recommender system we found two problems:

1. **Missing places:** 37 trips have no `place_slug`. If we keep them, "missing" becomes a place that could be recommended.
2. **Country instead of city:** about 9,400 trips only record the **country** (`united-states`, `spain`), not a real destination. A travel agency cannot sell "a trip to the United States", so we remove these rows as well.

In [3]:
print("Trips with no place:", trips["place_slug"].isna().sum())

country_only = trips["place_slug"] == trips["country_slug"]
print("Trips that only record a country:", country_only.sum())
trips.loc[country_only, "place_slug"].value_counts().head()

Trips with no place: 37
Trips that only record a country: 9412


place_slug
united-states     1703
spain              489
italy              388
france             360
united-kingdom     318
Name: count, dtype: int64

We remove both kinds of rows:

In [4]:
trips = trips.dropna(subset=["place_slug"])
trips = trips[trips["place_slug"] != trips["country_slug"]]

print("Trips left:", len(trips))
print("Travellers:", trips["username"].nunique())
print("Places:", trips["place_slug"].nunique())

Trips left: 37061
Travellers: 2789
Places: 731


After cleaning we keep **37,061 trips** by **2,789 travellers** to **731 real destinations**. Removing the country-only rows costs ~20 % of the trips  😢  , but every place left is a destination the agency can actually offer.

# 2. MATRIX - Traveler x Place



We turn the trips into a **traveller × place** table: `1` = the traveller has been there, `0` = not.

We only care *whether* someone went to a place, not how many times, so repeated visits count once.

In [5]:
matrix = (pd.crosstab(trips["username"], trips["place_slug"]) > 0).astype(int)

print("Shape:", matrix.shape)
print("Filled in:", round(matrix.values.mean() * 100, 2), "%")
#example 
matrix.iloc[:2, :8]

Shape: (2789, 731)
Filled in: 1.18 %


place_slug,a-coruna-spain,aachen-germany,aalborg-denmark,aalesund-norway,aarhus-denmark,abbotsford-canada,aberdeen-united-kingdom,abu-dhabi-united-arab-emirates
username,,,,,,,,
@01aniqe,0,0,0,0,0,0,0,0
@0chucha0,0,0,0,0,0,0,0,0


**Most of the matrix will have 0 s**

In [6]:
visits = matrix.sum(axis=0).sort_values(ascending=False)   # number of travellers per place
print("Places with at least 50 travellers:", (visits >= 50).sum())
print("Places with fewer than 5 travellers:", (visits < 5).sum())
visits.head(10)

Places with at least 50 travellers: 120
Places with fewer than 5 travellers: 255


place_slug
bangkok-thailand                  641
chiang-mai-thailand               561
london-united-kingdom             546
berlin-germany                    483
new-york-city-ny-united-states    445
barcelona-spain                   430
san-francisco-ca-united-states    426
paris-france                      395
amsterdam-netherlands             372
ubud-bali-indonesia               360
dtype: int64

In [7]:
print("Places with only 1 traveller:", (visits == 1).sum())
visits.tail(10)

Places with only 1 traveller: 100


place_slug
banja-luka-bosnia    1
bhubaneswar-india    1
ufa-russia           1
tripoli-libya        1
tomsk-russia         1
toluca-mexico        1
benghazi-libya       1
ulyanovsk-russia     1
abbotsford-canada    1
zhangjiakou-china    1
dtype: int64

The matrix is **99 % empty**: an average traveller has visited only a handful of the 731 places. Popularity is also very uneven: Bangkok has 641 travellers, while 255 places have fewer than 5. A place visited by only 2–3 people gives very weak evidence, so later we will only recommend places with enough travellers.

# 3.Similarity between places, ITEM BASED

**Why item-based and not user-based?** Both use the same trips, but item-based relies on **many travellers per place** (for example, 148 people visited both Lisbon and Berlin), while user-based relies on **one similar person**, who may share only 1–2 places with our customer. Place-to-place patterns are therefore more rational and less affected by one person's unusual trip.

In [8]:
place_sim = pd.DataFrame(cosine_similarity(matrix.T), index=matrix.columns, columns=matrix.columns)
print(place_sim.shape)

(731, 731)




Two places are similar when **the same travellers** visited both. We compare the columns of the matrix with **cosine similarity**: 1 = visited by exactly the same people, 0 = no traveller in common.
Cosine is a good choice here because it divides by each place's number of travellers, so a big hub does not look similar to everything just because it is big. It reduces the popularity effect, but does not remove it (see the limitations).

In [9]:
def similar_places(place, n=5, min_travellers=50):
    """The n places most similar to `place`, with the number of travellers as evidence."""
    sims = place_sim[place].drop(place)                         # drop the place itself
    sims = sims[visits[sims.index] >= min_travellers]           # keep only places with enough evidence
    top = sims.sort_values(ascending=False).head(n)
    return pd.DataFrame({"similarity": top.round(2), "travellers": visits[top.index]})

similar_places("lisbon-portugal")

,similarity,travellers
place_slug,,
porto-portugal,0.37,117
barcelona-spain,0.36,430
berlin-germany,0.36,483
london-united-kingdom,0.33,546
madrid-spain,0.32,213


The result makes sense: travellers who went to Lisbon also went to Porto, Barcelona, Berlin and Madrid. The algorithm found this **only from travel behaviour**, without knowing where Lisbon is. We only keep places with at least 50 travellers, so every similarity is backed by enough people.

# Actual Recommendation for a Client

## 4. Recommending a next destination for a customer

For a customer we take **all the places they have visited**. For every place they have *not* visited, we add up its similarity to each of their places. The place with the highest total score is the one most connected to their travel history, so it becomes the recommendation.
We only recommend places with at least 50 travellers, as before, so every recommendation is backed by "evidence".
- we also chose the one customer from the lecture.

In [10]:
customer = "@pfitznerrene"      
visited = matrix.columns[matrix.loc[customer] > 0]
print(customer, "has visited:", list(visited))

@pfitznerrene has visited: ['athens-greece', 'berlin-germany', 'lisbon-portugal']


 - Recommendation Function

In [11]:
def recommend(customer, n=5, min_travellers=50):
    """Top-n places for a customer, based on all the places they have visited."""
    visited = matrix.columns[matrix.loc[customer] > 0]
    candidates = [p for p in matrix.columns if p not in visited and visits[p] >= min_travellers]
    scores = place_sim.loc[candidates, visited].sum(axis=1)       # add up similarity to each visited place
    top = scores.sort_values(ascending=False).head(n)
    return pd.DataFrame({"score": top.round(2), "travellers": visits[top.index]})

recommend(customer)

,score,travellers
place_slug,,
barcelona-spain,0.99,430
amsterdam-netherlands,0.98,372
london-united-kingdom,0.98,546
budapest-hungary,0.90,296
paris-france,0.88,395


- Another customer: @klavs

In [12]:
customer2 = "@klavs"      
print(customer2, "has visited:", list(matrix.columns[matrix.loc[customer2] > 0]))
recommend(customer2)

@klavs has visited: ['bangkok-thailand', 'chiang-mai-thailand']


,score,travellers
place_slug,,
kuala-lumpur-malaysia,0.85,345
singapore-singapore,0.84,347
ho-chi-minh-city-vietnam,0.78,264
phuket-thailand,0.75,184
ubud-bali-indonesia,0.73,360


# 5 . Why would the customer be interested? (statistically)

For the top recommendation (Barcelona) we show **where the score comes from**: for each place the customer has visited, how similar it is to the recommended place and how many travellers went to both.

In [13]:
def explain(customer, place):
    """Why `place` is recommended: its link to every place the customer has visited."""
    visited = matrix.columns[matrix.loc[customer] > 0]
    return pd.DataFrame({
        "similarity": place_sim.loc[place, visited].round(2),
        "travelled_to_both": [(matrix[place] & matrix[v]).sum() for v in visited],
        "travellers_of_visited_place": visits[visited]})

explain("@pfitznerrene", "barcelona-spain")

,similarity,travelled_to_both,travellers_of_visited_place
place_slug,,,
athens-greece,0.23,56,141
berlin-germany,0.40,181,483
lisbon-portugal,0.36,142,356


**Are these similarities low?** No. Cosine similarity between two places = travellers who visited both ÷ √(travellers of A × travellers of B). For Berlin and Barcelona: 181 / √(483 × 430) = 0.40. Because travellers go to very different places, a typical pair of places (among those with 50+ travellers) scores only about **0.10**, and fewer than 1 % of pairs reach 0.40. Berlin–Barcelona is in fact the strongest link Barcelona has.

# 6. Final

### Conclusion

**Customer @pfitznerrene** – Athens, Berlin, Lisbon → **Barcelona**.
181 travellers who visited Berlin, 142 who visited Lisbon and 56 who visited Athens also went to Barcelona: a popular European city, just like his previous trips.



### Why item-based filtering?
It is based on **real behaviour** (which places travellers actually combine), uses the customer's **whole history**, and every offer comes with **evidence** ("181 travellers went to both"), which makes it easy to explain.

### Limitations
**Popularity bias** (big hubs like Barcelona are recommended often), **cold start** (no trips = no recommendation) .